# Notebook 05: LangGraph Agent

Same agent using LangGraph framework for comparison.

## 1. Setup

In [2]:
# Install if needed
!pip install langgraph langchain-openai langchain-core

     |████████████████████████████████| 155 kB 1.9 MB/s eta 0:00:01
     |████████████████████████████████| 75 kB 12.8 MB/s eta 0:00:01
     |████████████████████████████████| 458 kB 1.3 MB/s eta 0:00:01
  Using cached xxhash-3.6.0-cp39-cp39-macosx_11_0_arm64.whl (30 kB)
     |████████████████████████████████| 56 kB 7.4 MB/s  eta 0:00:01
     |████████████████████████████████| 45 kB 8.0 MB/s  eta 0:00:01
     |████████████████████████████████| 1.1 MB 1.3 MB/s eta 0:00:01
     |████████████████████████████████| 997 kB 83.2 MB/s eta 0:00:01
     |████████████████████████████████| 396 kB 1.3 MB/s eta 0:00:01
     |████████████████████████████████| 601 kB 48.5 MB/s eta 0:00:01
  Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
  Using cached packaging-25.0-py3-none-any.whl (66 kB)
     |████████████████████████████████| 367 kB 46.0 MB/s eta 0:00:01
     |████████████████████████████████| 245 kB 79.0 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 5.8 MB/s  eta 0:00:01
  

In [1]:
import os
import json
import random
from typing import Dict, Annotated, TypedDict
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
import boto3

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
KNOWLEDGE_BASE_ID = os.getenv('KNOWLEDGE_BASE_ID')

bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

print('[OK] Configuration loaded')

[OK] Configuration loaded


## 2. Mock Data

In [2]:
MOCK_ORDERS = {
    "ORD-12345": {"status": "shipped", "carrier": "FedEx", "estimated_delivery": "2026-02-03", 
                  "items": ["Blue iPhone 15 Case"], "destination": "New York, NY"},
    "ORD-67890": {"status": "processing", "estimated_delivery": "2026-02-05", 
                  "items": ["Wireless Earbuds"], "destination": "Miami, FL"}
}

MOCK_WEATHER = {
    "miami": {"condition": "Hurricane Warning", "delay_days": 3},
    "new york": {"condition": "Clear", "delay_days": 0}
}

MOCK_INVENTORY = {
    "iphone 15 case": {"blue": {"in_stock": True, "quantity": 42}, "black": {"in_stock": False}},
    "airpods pro": {"default": {"in_stock": True, "quantity": 120}}
}

print('[OK] Mock data loaded')

[OK] Mock data loaded


## 3. Define Tools with @tool Decorator

In [3]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search company knowledge base for policies, FAQs, shipping info, and general questions."""
    try:
        response = bedrock_agent.retrieve(
            knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={'text': query},
            retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
        )
        results = [r['content']['text'] for r in response.get('retrievalResults', [])]
        return json.dumps({'success': True, 'results': results})
    except Exception as e:
        return json.dumps({'success': False, 'error': str(e)})

@tool
def check_order_status(order_id: str) -> str:
    """Check order status, tracking, and delivery ETA. Requires order ID."""
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id in MOCK_ORDERS:
        return json.dumps({'success': True, 'order_id': order_id, **MOCK_ORDERS[order_id]})
    return json.dumps({'success': False, 'error': f'Order {order_id} not found'})

@tool
def get_weather_alerts(location: str) -> str:
    """Check weather alerts and shipping delays for a location. Use when customer asks about weather impact on deliveries or delays to a city."""
    key = location.lower().split(',')[0].strip()
    if key in MOCK_WEATHER:
        data = MOCK_WEATHER[key]
        return json.dumps({'success': True, 'location': location, **data})
    return json.dumps({'success': True, 'location': location, 'condition': 'Clear', 'delay_days': 0})

@tool
def check_inventory(product_name: str, color: str = None) -> str:
    """Check if a product is in stock and available quantities."""
    key = product_name.lower().strip()
    for pkey in MOCK_INVENTORY:
        if pkey in key or key in pkey:
            data = MOCK_INVENTORY[pkey]
            if color and color.lower() in data:
                return json.dumps({'success': True, 'product': pkey.title(), **data[color.lower()]})
            return json.dumps({'success': True, 'product': pkey.title(), 'variants': data})
    return json.dumps({'success': False, 'error': 'Product not found'})

@tool
def create_return_request(order_id: str, reason: str) -> str:
    """Create a return request for an order. Customer must provide order ID and reason."""
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"): order_id = f"ORD-{order_id}"
    if order_id not in MOCK_ORDERS:
        return json.dumps({'success': False, 'error': 'Order not found'})
    return_id = f"RET-{random.randint(10000, 99999)}"
    return json.dumps({'success': True, 'return_id': return_id, 'order_id': order_id})

tools = [search_knowledge_base, check_order_status, get_weather_alerts, check_inventory, create_return_request]
print(f'[OK] {len(tools)} tools defined')

[OK] 5 tools defined


## 4. Initialize LLM with Tools

In [4]:
llm = ChatOpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, model=LLM_MODEL, temperature=0.1)
llm_with_tools = llm.bind_tools(tools)
print('[OK] LLM initialized with tools')

[OK] LLM initialized with tools


## 5. Define Agent State and Nodes

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

SYSTEM_PROMPT = """You are a helpful e-commerce customer service agent.

Available tools:
- search_knowledge_base: For policies, FAQs, general questions
- check_order_status: For order tracking (requires order ID)
- get_weather_alerts: For weather-related shipping delays (just needs city name)
- check_inventory: For stock availability
- create_return_request: To process returns (requires order ID and reason)

IMPORTANT: Be proactive - use tools immediately when relevant. For weather/delay questions about a city, call get_weather_alerts directly with the city name."""

def agent_node(state: AgentState) -> AgentState:
    """Agent node - calls LLM to decide action."""
    messages = state['messages']
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

tool_node = ToolNode(tools)

def should_continue(state: AgentState) -> str:
    """Route to tools or end."""
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return 'tools'
    return END

print('[OK] Nodes defined')

[OK] Nodes defined


## 6. Build and Compile Graph

In [6]:
workflow = StateGraph(AgentState)
workflow.add_node('agent', agent_node)
workflow.add_node('tools', tool_node)
workflow.set_entry_point('agent')
workflow.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})
workflow.add_edge('tools', 'agent')
app = workflow.compile()

print('[OK] Graph compiled')
print('Structure: START -> agent -> (tools?) -> agent -> END')

[OK] Graph compiled
Structure: START -> agent -> (tools?) -> agent -> END


## 7. Test Helper Function

In [7]:
def run_agent(query: str, verbose: bool = True) -> str:
    print("=" * 60)
    print(f"Query: {query}")
    print("=" * 60)
    
    result = app.invoke({'messages': [HumanMessage(content=query)]})
    
    if verbose:
        tools_called = []
        for msg in result['messages']:
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                tools_called.extend([tc['name'] for tc in msg.tool_calls])
        if tools_called:
            print(f"Tools: {tools_called}")
    
    print(f"\nResponse: {result['messages'][-1].content}")
    return result['messages'][-1].content

## 8. Test: Order Status

In [8]:
run_agent("What is the status of order ORD-12345?")

Query: What is the status of order ORD-12345?
Tools: ['check_order_status']

Response: Your order with ID **ORD-12345** has been shipped via FedEx. The estimated delivery date is **February 3, 2026**. The item included in this order is a Blue iPhone 15 Case, and it is being delivered to New York, NY.


'Your order with ID **ORD-12345** has been shipped via FedEx. The estimated delivery date is **February 3, 2026**. The item included in this order is a Blue iPhone 15 Case, and it is being delivered to New York, NY.'

## 9. Test: Weather Delay

In [9]:
run_agent("Will my package to Miami be delayed due to weather?")

Query: Will my package to Miami be delayed due to weather?
Tools: ['get_weather_alerts']

Response: Yes, your package to Miami is likely to be delayed due to a Hurricane Warning in the area. The expected delay is approximately 3 days. Please stay safe and let me know if there's anything else I can assist you with.


"Yes, your package to Miami is likely to be delayed due to a Hurricane Warning in the area. The expected delay is approximately 3 days. Please stay safe and let me know if there's anything else I can assist you with."

## 10. Test: Inventory

In [10]:
run_agent("Is the blue iPhone 15 case in stock?")

Query: Is the blue iPhone 15 case in stock?
Tools: ['check_inventory']

Response: Yes, the blue iPhone 15 case is in stock, with 42 units available.


'Yes, the blue iPhone 15 case is in stock, with 42 units available.'

## 11. Test: Create Return

In [11]:
run_agent("I want to return order ORD-12345 because it's defective.")

Query: I want to return order ORD-12345 because it's defective.
Tools: ['create_return_request']

Response: Your return request for order **ORD-12345** has been successfully processed due to the item being defective. Your return ID is **RET-31605**. If you need further assistance, feel free to ask!


'Your return request for order **ORD-12345** has been successfully processed due to the item being defective. Your return ID is **RET-31605**. If you need further assistance, feel free to ask!'

## 12. Test: Multi-Tool Query

In [12]:
run_agent("Check order ORD-67890 and tell me if it will be delayed due to Miami weather.")

Query: Check order ORD-67890 and tell me if it will be delayed due to Miami weather.
Tools: ['check_order_status', 'get_weather_alerts']

Response: Your order (ORD-67890) is currently processing and is estimated to be delivered by February 5, 2026. However, due to a hurricane warning in Miami, there is a potential delay of 3 days. Please keep an eye on updates for any changes in the delivery schedule.


'Your order (ORD-67890) is currently processing and is estimated to be delivered by February 5, 2026. However, due to a hurricane warning in Miami, there is a potential delay of 3 days. Please keep an eye on updates for any changes in the delivery schedule.'

## 13. Add Memory (Checkpointing)

In [13]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
app_with_memory = workflow.compile(checkpointer=memory)

config = {'configurable': {'thread_id': 'user-123'}}

print("Turn 1:")
r1 = app_with_memory.invoke({'messages': [HumanMessage(content="What is the status of order ORD-12345?")]}, config)
print(r1['messages'][-1].content)

print("\nTurn 2 (follow-up):")
r2 = app_with_memory.invoke({'messages': [HumanMessage(content="When will it arrive?")]}, config)
print(r2['messages'][-1].content)

Turn 1:
Your order with ID **ORD-12345** has been shipped via FedEx. The estimated delivery date is February 3, 2026. The order includes a Blue iPhone 15 Case and is being delivered to New York, NY.

Turn 2 (follow-up):
Your order is estimated to arrive on February 3, 2026.


## 14. Comparison: Manual vs LangGraph

| Aspect | Manual | LangGraph |
|--------|--------|----------|
| Code lines | ~100 | ~50 |
| Tool definition | Dict schema | @tool decorator |
| State management | Manual | TypedDict + graph |
| Loop control | while True | Graph edges |
| Memory | Build yourself | Built-in checkpointing |
| Debugging | Print statements | State inspection |

**Use Manual:** Learning, simple apps, minimal dependencies

**Use LangGraph:** Production, complex workflows, multi-agent systems